In [1]:
import torchvision
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score
import torch.nn as nn
from torch import optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import re
import shutil
import random

## Data preprocessing

In [10]:
# Replace last classifier to only handle 2 cases, one benign one high grade
num_classes = 2
model = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)

# Modify the classifier
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, num_classes)

# Freeze all layers except the classifier
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

labels = pd.read_csv('case_grade_match.csv')

In [11]:
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        if os.path.getsize(os.path.join(patch_dir, filename)) < 2000:
            continue
        if 'patched_' in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

patches = group_patches('filtered_patches/')
case_nums = list(patches.keys())
dataset = labels.loc[[(int(x)-1) for x in case_nums]]
noindex = dataset.Class != 2.0
X = dataset[noindex].Case
y = dataset[noindex].Class
train, test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=40)

train_patches = {case_num: patches[int(case_num)] for case_num in train}
test_patches = {case_num: patches[int(case_num)] for case_num in test}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = PNGDataset(train_patches, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)



## Train model, 5 epochs

In [12]:
torch.device('mps')

device(type='mps')

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("mps")

# Load pre-trained DenseNet
model = models.densenet121(pretrained=True)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2)  # 2 classes: low grade (0) and high grade (1)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training function
def train_model(model, train_loader, criterion, optimizer, num_epochs=5):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

# Evaluation function
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    print(f"Test Accuracy: {100*correct/total:.2f}%")

# Train and evaluate
train_model(model, train_dataloader, criterion, optimizer, num_epochs=5)
evaluate_model(model, test_dataloader)


/Users/margaretpirozzolo/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/margaretpirozzolo/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/5, Loss: 0.4412, Accuracy: 80.68%
Epoch 2/5, Loss: 0.3286, Accuracy: 85.84%
Epoch 3/5, Loss: 0.2724, Accuracy: 88.76%
Epoch 4/5, Loss: 0.2267, Accuracy: 90.80%
Epoch 5/5, Loss: 0.1873, Accuracy: 92.30%
Test Accuracy: 77.03%


In [ ]:
pred = []
labels = []
with torch.no_grad():
    for images, label in test_dataloader:
        images, label = images.to(device), label.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        labels.append(label)
        pred.append(predicted)

pred = torch.cat(pred).cpu()
labels = torch.cat(labels).cpu()
accuracy = accuracy_score(labels, pred)
print(f'Accuracy: {accuracy}')
confusion_matrix(labels, pred)

## Val, train and test data Plus Early Stopping

In [8]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms

# Function to group patches by case number
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        if os.path.getsize(os.path.join(patch_dir, filename)) < 2000:
            continue
        if 'patched_' in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

# Custom Dataset class
class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Load patches
dataset_dir = 'filtered_patches/'
patches = group_patches(dataset_dir)

# Load labels
labels = pd.read_csv('case_grade_match.csv')

# Filter valid cases
case_nums = list(patches.keys())
dataset = labels.loc[labels['Case'].isin(case_nums)]
noindex = dataset['Class'] != 2.0
filtered_dataset = dataset[noindex]

# Split into train, validation, and test sets
train_cases, test_cases, y_train, y_test = train_test_split(
    filtered_dataset['Case'], filtered_dataset['Class'], test_size=0.2, stratify=filtered_dataset['Class'], random_state=40)
train_cases, val_cases, y_train, y_val = train_test_split(
    train_cases, y_train, test_size=0.2, stratify=y_train, random_state=40)

# Ensure no overlap in cases across datasets
train_patches = {case_num: patches[case_num] for case_num in train_cases}
val_patches = {case_num: patches[case_num] for case_num in val_cases}
test_patches = {case_num: patches[case_num] for case_num in test_cases}

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = PNGDataset(train_patches, labels, transform=transform)
val_dataset = PNGDataset(val_patches, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform=transform)

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

def evaluate_model(model, data_loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def train_model(model, train_loader, val_loader, test_loader, criterion, optimizer, num_epochs=50):
    best_loss = np.inf
    patience = 3
    counter = 0
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)
        
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered.")
                break
    
    # Final evaluation on test dataset
    test_loss, test_acc = evaluate_model(model, test_loader, criterion)
    print(f"Final Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%")


# Train with early stopping
train_model(model, train_dataloader, val_dataloader, test_dataloader, criterion, optimizer, num_epochs=50)


Epoch 1/50, Train Loss: 0.2621, Train Acc: 89.49%, Val Loss: 0.4725, Val Acc: 84.66%
Epoch 2/50, Train Loss: 1.3813, Train Acc: 73.43%, Val Loss: 0.4470, Val Acc: 84.34%
Epoch 3/50, Train Loss: 0.5586, Train Acc: 75.38%, Val Loss: 0.4627, Val Acc: 84.34%
Epoch 4/50, Train Loss: 0.5577, Train Acc: 75.38%, Val Loss: 0.4588, Val Acc: 84.34%
Epoch 5/50, Train Loss: 0.5556, Train Acc: 75.38%, Val Loss: 0.4604, Val Acc: 84.34%
Early stopping triggered.
Final Test Loss: 0.6262, Test Accuracy: 68.33%


In [9]:
pred = []
labels = []
with torch.no_grad():
    for images, label in test_dataloader:
        images, label = images.to(device), label.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        labels.append(label)
        pred.append(predicted)

pred = torch.cat(pred).cpu()
labels = torch.cat(labels).cpu()
accuracy = accuracy_score(labels, pred)
print(f'Accuracy: {accuracy}')
confusion_matrix(labels, pred)

Accuracy: 0.6833375219904498


array([[   0, 1260],
       [   0, 2719]])